<a href="https://colab.research.google.com/github/muhammeddanisht/langgraph-agents/blob/main/langgraph_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Packages

In [1]:
!pip install -q fastmcp langchain-mcp-adapters langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.2/221.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.9/272.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires sta

 Imports

In [ ]:
# MCP server builder
from fastmcp import FastMCP

# For running server in background (Colab fix)
import threading
import time

# For real weather data — no API key needed
import requests

# Groq free LLM
from langchain_groq import ChatGroq

# MCP client — connects agent to MCP server
from langchain_mcp_adapters.client import MultiServerMCPClient

# Agent builder
from langgraph.prebuilt import create_react_agent

# Message format
from langchain_core.messages import HumanMessage

from langchain.agents import create_agent

Get API Key from Colab Secrets

In [ ]:
import os
from google.colab import userdata

# Read key from Colab secrets safe
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API key loaded from Colab secrets!")

✅ API key loaded from Colab secrets!


 Build MCP Server with Real Tools

In [ ]:
# Step 1 — Create the MCP server
# Give it a name — "my_tools"
mcp = FastMCP("my_tools")

Calculator Tool

In [ ]:
# @mcp.tool() = name tag that registers this as an MCP tool

@mcp.tool()
def calculator(expression: str) -> str:
    """
    Calculate a math expression and return the result.
    Example: '25 * 4' or '100 / 5' or '10 + 3 - 2'
    """
    try:
        # eval() runs real Python math — not fake, not hardcoded
        result = eval(expression)
        return f"The result of {expression} = {result}"
    except Exception as e:
        return f"Math error: {str(e)}"

 Weather Tool (real live data from wttr.in)

In [ ]:
@mcp.tool()
def get_weather(city: str) -> str:
    """
    Get real current weather for any city in the world.
    Example: 'London' or 'Tokyo' or 'Kozhikode'
    """
    try:
        # wttr.in = free weather service, no API key needed
        # format=3 gives us: "City: emoji temperature"
        url = f"https://wttr.in/{city}?format=3"
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            return f"Current weather → {response.text.strip()}"
        else:
            return f"Could not fetch weather for {city}"
    except Exception as e:
        return f"Weather error: {str(e)}"

 Start server in background thread (Colab fix)

In [ ]:
# daemon=True means: when notebook stops, server stops too

def run_server():
    mcp.run(transport="streamable-http")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give server 2 seconds to fully start before we connect
time.sleep(2)

print("✅ MCP Server running in background!")
print("📡 Listening at: http://localhost:8000/mcp")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.4.2                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      my_tools, 3.4.2                             │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[06/11/26 22:31:55] INFO     Starting MCP server 'my_tools' with transport 'streamable-http' on    ]8;id=852911;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=447303;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#304\304]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [28755]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✅ MCP Server running in background!
📡 Listening at: http://localhost:8000/mcp


Build the LLM (Groq free model)

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("✅ Groq LLM ready!")

✅ Groq LLM ready!


  asyncio for Colab

In [ ]:

!pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

print("✅ asyncio Colab fix applied!")

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='functools' coro=<AsyncIOBackend.run.<locals>.wrapper() done, defined at /usr/local/lib/python3.12/dist-packages/anyio/_backends/_asyncio.py:2334> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 172, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1584, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): [errno 98] address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File

✅ asyncio Colab fix applied!


Connect Agent to MCP Server (Modern 2026 way)


In [ ]:
from langchain.agents import create_agent

async def build_agent():
    client = MultiServerMCPClient(
        {
            "my_tools": {
                "transport": "http",
                "url": "http://localhost:8000/mcp"
            }
        }
    )

    tools = await client.get_tools()

    # Modern 2026 way — no deprecation warning
    agent = create_agent(
        model=llm,
        tools=tools
    )

    return agent, tools

agent, tools = asyncio.run(build_agent())

print(f"✅ Agent ready!")
print(f"🔧 Tools loaded: {[t.name for t in tools]}")

INFO:     127.0.0.1:50696 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50712 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50722 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50738 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50752 - "DELETE /mcp HTTP/1.1" 200 OK
✅ Agent ready!
🔧 Tools loaded: ['calculator', 'get_weather']


Test the Agent


In [ ]:


import asyncio

async def ask_agent(question):
    print(f"\n❓ Question: {question}")
    print("-" * 50)

    response = await agent.ainvoke({
        "messages": [HumanMessage(content=question)]
    })

    # Get the last message — that's the final answer
    final_answer = response["messages"][-1].content
    print(f"🤖 Answer: {final_answer}")
    return final_answer

# Test 1 — Calculator (real math)
asyncio.run(ask_agent("What is 347 multiplied by 28?"))


❓ Question: What is 347 multiplied by 28?
--------------------------------------------------
INFO:     127.0.0.1:50756 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50760 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50758 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50774 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50776 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50782 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: The result of the multiplication is 9716.


'The result of the multiplication is 9716.'

Test Weather Tool

In [ ]:
asyncio.run(ask_agent("What is the current weather in Tokyo?"))


❓ Question: What is the current weather in Tokyo?
--------------------------------------------------
INFO:     127.0.0.1:50794 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50816 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50802 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50822 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50832 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50834 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: The current weather in Tokyo is 21°C with sunny conditions.


'The current weather in Tokyo is 21°C with sunny conditions.'

In [ ]:
# Boss Test (no hints given)

asyncio.run(ask_agent("I have 3 boxes. Each box has 144 items. How many items total?"))


❓ Question: I have 3 boxes. Each box has 144 items. How many items total?
--------------------------------------------------
INFO:     127.0.0.1:50838 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50858 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50854 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50866 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50876 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50878 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: You have 432 items in total.


'You have 432 items in total.'

In [ ]:
#— Another boss test

asyncio.run(ask_agent("What's the weather in Kozhikode right now?"))


❓ Question: What's the weather in Kozhikode right now?
--------------------------------------------------
INFO:     127.0.0.1:50892 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50904 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50910 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50918 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50928 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50938 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: The current weather in Kozhikode is 25°C with rain.


'The current weather in Kozhikode is 25°C with rain.'